<a href="https://colab.research.google.com/github/Luseat/automatic_dataset_labeling/blob/main/labeled_dataset_from_hugging_face.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets torch torchvision tqdm

In [ ]:
import os

emotions = ['Happy', 'Sad', 'Angry', 'Fear', 'Surprise', 'Disgust', 'Neutral']
os.makedirs('labeled_hf', exist_ok=True)

for em in emotions:
  os.makedirs(f'labeled_hf/{em}', exist_ok=True)

print("Folder berhasil dibuat...")

Folder berhasil dibuat...


In [ ]:
from transformers import pipeline
from datasets import load_dataset
import os
from tqdm import tqdm


print("Mempersiapkan AI SigLIP...")
classifier = pipeline("zero-shot-image-classification", model="google/siglip-base-patch16-224", device=0)


labels = [
    "anime face smiling happy",
    "anime face crying sad",
    "anime face angry mad",
    "anime face scared fear",
    "anime face surprised shocked",
    "anime face disgusted gross",
    "anime face neutral expressionless"
]


label_map = {
    "anime face smiling happy": "Happy",
    "anime face crying sad": "Sad",
    "anime face angry mad": "Angry",
    "anime face scared fear": "Fear",
    "anime face surprised shocked": "Surprise",
    "anime face disgusted gross": "Disgust",
    "anime face neutral expressionless": "Neutral"
}


# Download dataset Hugging Face
print("Mehubungkan ke Hugging Face...")
dataset = load_dataset("miaojiemiao/Anime-2026", split="train", streaming=True)


TARGET_PER_CLASS = 5000
counts = {em: 0 for em in label_map.values()}
img_counter = 0 # Buat ngasih nama file yang unik ajh sih


print("Mulai untuk pernyotiran jutaan gambar dari Cloud...")
for data in tqdm(dataset):
  if all (v >= TARGET_PER_CLASS for v in counts.values()):
    print("\nTarget Gambar Sudah Tercapai!")
    break


  try:
    img = data['jpg'].convert('RGB') # Di Hugging Face, gambarnya biasa disimpen di kolom bernama 'image'
    preds = classifier(img, candidate_labels = labels)
    top_pred = preds[0]


    if top_pred['score'] > 0.05:
      emotion = label_map[top_pred['label']]
      if counts[emotion] < TARGET_PER_CLASS:
        img_name = f"anime_hd_{img_counter}.jpg" # Generate nama file baru
        desth_path = os.path.join('labeled_hf', emotion, img_name)


        img.save(desth_path)


        counts[emotion] += 1
        img_counter += 1


  except Exception as e:
    continue


print("Done, jumlah per emosi:", counts)

Mempersiapkan AI SigLIP...


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Mehubungkan ke Hugging Face...


Resolving data files:   0%|          | 0/190 [00:00<?, ?it/s]

Mulai untuk pernyotiran jutaan gambar dari Cloud...


13448it [9:00:01,  2.31s/it]

**Ini buat ngeliat aja Score sebenernya berapa**

In [ ]:
from transformers import pipeline
from datasets import load_dataset

print("Mempersiapkan AI SigLIP...")
classifier = pipeline("zero-shot-image-classification", model="google/siglip-base-patch16-224", device=0)
dataset = load_dataset("miaojiemiao/Anime-2026", split="train", streaming=True)

labels = [
    "anime face smiling happy",
    "anime face crying sad",
    "anime face angry mad",
    "anime face scared fear",
    "anime face surprised shocked",
    "anime face disgusted gross",
    "anime face neutral expressionless"
]

print("Ngintip skor SigLIP...")
for i, data in enumerate(dataset):
    # kuncinya: ganti dari ['image'] jadi ['jpg']
    img = data['jpg'].convert('RGB')
    preds = classifier(img, candidate_labels=labels)

    print(f"Gambar {i+1} -> Emosi: {preds[0]['label']} | Skor: {preds[0]['score']}")

    # Cuma ngetes 10 gambar doang
    if i == 9:
        break

Mempersiapkan AI SigLIP...


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/190 [00:00<?, ?it/s]

Ngintip skor SigLIP...
Gambar 1 -> Emosi: anime face disgusted gross | Skor: 0.012923428788781166
Gambar 2 -> Emosi: anime face smiling happy | Skor: 0.0002726610691752285
Gambar 3 -> Emosi: anime face smiling happy | Skor: 0.0036474198568612337
Gambar 4 -> Emosi: anime face crying sad | Skor: 0.19884026050567627
Gambar 5 -> Emosi: anime face angry mad | Skor: 8.416379273512575e-07
Gambar 6 -> Emosi: anime face smiling happy | Skor: 0.00015839831030461937
Gambar 7 -> Emosi: anime face smiling happy | Skor: 6.555103027494624e-05
Gambar 8 -> Emosi: anime face crying sad | Skor: 0.0001426349044777453
Gambar 9 -> Emosi: anime face disgusted gross | Skor: 0.0008480816031806171
Gambar 10 -> Emosi: anime face scared fear | Skor: 0.00023423497623298317
